# Neuron Frequency Distribution — Stacked Bar

Per-neuron stacked bar chart: each bar is one neuron, each segment is one frequency's
fraction of that neuron's Fourier norm. Segments below `threshold` collapse into grey.

Use `variant.at(epoch).view("activations.mlp.neuron_freq_distribution")` — a registered
view using the same `neuron_freq_norm` artifact as the frequency cluster heatmap.

Key kwargs: `threshold` (default 0.10), `sort_by` (default `"dominant"`).

## Parameters

In [ ]:
FAMILY    = "modulo_addition_1layer"
THRESHOLD = 0.10
HEIGHT    = 500
WIDTH     = 1100

# Variants to sweep — (prime, model_seed, data_seed)
VARIANTS = [
    (101, 999, 598),   # healthy
    (101, 999,  42),   # late_grokker
    (101, 999, 999),   # degraded
]

## Setup

In [2]:
import json
from miscope import load_family

family = load_family(FAMILY)


def key_epochs_from_summary(variant) -> list[int]:
    """Derive key display epochs from the variant's summary JSON.

    Returns the window boundary epochs that mark meaningful transitions:
    pre-competition, plateau onset, second descent onset, cascade (if any),
    and final epoch.
    """
    summary_path = variant.variant_dir / "variant_summary.json"
    s = json.loads(summary_path.read_text())

    window_names = [
        "first_descent_window",
        "plateau_window",
        "cascade_window",
        "second_descent_window",
        "final_window",
    ]

    boundaries = {500}  # always include pre-competition
    for wname in window_names:
        w = s.get(wname, {})
        if w.get("skipped"):
            continue
        boundaries.add(w["start_epoch"])
        boundaries.add(w["end_epoch"])

    # Snap each boundary to the nearest available artifact epoch
    available = set(variant.get_artifact_loader().get_epochs("neuron_freq_norm"))
    snapped = sorted(
        min(available, key=lambda e: abs(e - b))
        for b in boundaries
    )
    return list(dict.fromkeys(snapped))  # deduplicate, preserve order


print("Key epochs per variant:")
for prime, seed, data_seed in VARIANTS:
    v = family.get_variant(prime=prime, seed=seed, data_seed=data_seed)
    epochs = key_epochs_from_summary(v)
    print(f"  p={prime} s={seed} ds={data_seed}: {epochs}")

Key epochs per variant:
  p=113 s=999 ds=598: [0, 500, 1200, 9500, 10700, 11600, 12400, 24999]
  p=113 s=999 ds=42: [0, 500, 1200, 20000, 22500, 23000, 24999]
  p=113 s=999 ds=999: [0, 500, 1200, 10000, 11600, 12200, 13000, 24999]


## Sweep — Key Epochs per Variant

For each variant, show the neuron frequency distribution at every window boundary
derived from the summary file.

In [3]:
VIEW = "activations.mlp.neuron_freq_distribution"

for prime, seed, data_seed in VARIANTS:
    v = family.get_variant(prime=prime, seed=seed, data_seed=data_seed)
    epochs = key_epochs_from_summary(v)
    print(f"\n── p={prime}  seed={seed}  data_seed={data_seed} ──")
    for epoch in epochs:
        v.at(epoch).view(VIEW).show(threshold=THRESHOLD, height=HEIGHT, width=WIDTH)


── p=113  seed=999  data_seed=598 ──



── p=113  seed=999  data_seed=42 ──



── p=113  seed=999  data_seed=999 ──


## Single Variant / Single Epoch

Quick one-off cell for inspecting a specific moment.

In [4]:
PRIME, SEED, DATA_SEED = 113, 999, 598
EPOCH = None   # None → final available epoch

v = family.get_variant(prime=PRIME, seed=SEED, data_seed=DATA_SEED)
if EPOCH is None:
    EPOCH = v.get_artifact_loader().get_epochs("neuron_freq_norm")[-1]

v.at(EPOCH).view(VIEW).show(threshold=THRESHOLD, height=HEIGHT, width=WIDTH)

## Export

Save PNGs for all key epochs across all VARIANTS.

In [5]:
from pathlib import Path

OUT_DIR = Path("exports")
OUT_DIR.mkdir(exist_ok=True)

for prime, seed, data_seed in VARIANTS:
    v = family.get_variant(prime=prime, seed=seed, data_seed=data_seed)
    epochs = key_epochs_from_summary(v)
    for epoch in epochs:
        fname = OUT_DIR / f"modulo_addition_1layer_p{prime}_seed{seed}_dseed{data_seed}epoch_{epoch}_neuron_freq_dist.png"
        v.at(epoch).view(VIEW).export("png", fname, threshold=THRESHOLD, height=HEIGHT, width=WIDTH)
        print(f"  saved {fname.name}")

  saved modulo_addition_1layer_p113_seed999_dseed598epoch_0_neuron_freq_dist.png
  saved modulo_addition_1layer_p113_seed999_dseed598epoch_500_neuron_freq_dist.png
  saved modulo_addition_1layer_p113_seed999_dseed598epoch_1200_neuron_freq_dist.png
  saved modulo_addition_1layer_p113_seed999_dseed598epoch_9500_neuron_freq_dist.png
  saved modulo_addition_1layer_p113_seed999_dseed598epoch_10700_neuron_freq_dist.png
  saved modulo_addition_1layer_p113_seed999_dseed598epoch_11600_neuron_freq_dist.png
  saved modulo_addition_1layer_p113_seed999_dseed598epoch_12400_neuron_freq_dist.png
  saved modulo_addition_1layer_p113_seed999_dseed598epoch_24999_neuron_freq_dist.png
  saved modulo_addition_1layer_p113_seed999_dseed42epoch_0_neuron_freq_dist.png
  saved modulo_addition_1layer_p113_seed999_dseed42epoch_500_neuron_freq_dist.png
  saved modulo_addition_1layer_p113_seed999_dseed42epoch_1200_neuron_freq_dist.png
  saved modulo_addition_1layer_p113_seed999_dseed42epoch_20000_neuron_freq_dist.png